# Tratamento dos Registros de Ocorrência do ISP-RJ

Este notebook reproduz a passagem dos Registros de Ocorrência ao total anual de
Mortes Violentas Intencionais (MVI) por bairro. O procedimento segue a ordem
apresentada nos capítulos 4 e 5 do TCC: seleção dos delitos, avaliação da
informação territorial, normalização das grafias, pareamento por Levenshtein e
Jaro-Winkler, aplicação das decisões manuais e compatibilização territorial.

O arquivo de entrada contém somente os registros e campos necessários ao recorte
de 2019 a 2024. Os dados são restritos e não devem ser redistribuídos fora do
repositório privado autorizado.


In [1]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd

PROJETO = Path.cwd().resolve()
if PROJETO.name == "01_preparacao_dados":
    PROJETO = PROJETO.parent

DADOS = PROJETO / "dados"
ARQUIVO_ISP = DADOS / "brutos" / "isp" / "isp_ro_mvi_2019_2024.csv"
ARQUIVO_BAIRROS = DADOS / "brutos" / "malhas" / "bairros_oficiais.csv"
ARQUIVO_CORRECOES = DADOS / "brutos" / "isp" / "regras_correcao_manual_bairros.xlsx"
ARQUIVO_REGRAS_REVISADAS = DADOS / "brutos" / "isp" / "regras_correcao_bairros_isp.csv"
ARQUIVO_REGISTROS_REVISADOS = DADOS / "brutos" / "isp" / "isp_registros_mvi_revisados_2019_2024.csv"
ARQUIVO_REGRAS = DADOS / "regras_compatibilizacao_bairros.csv"
SAIDA = DADOS / "intermediarios" / "isp_bairro_ano.csv"
SAIDA_REGISTROS = DADOS / "intermediarios" / "isp_registros_mvi_tratados.csv"
SAIDA_AUDITORIA = PROJETO / "resultados" / "auditorias" / "isp_tratamento.json"

SAIDA.parent.mkdir(parents=True, exist_ok=True)
SAIDA_AUDITORIA.parent.mkdir(parents=True, exist_ok=True)


## Funções de normalização e comparação textual


In [2]:
def normalizar_texto(valor):
    if pd.isna(valor):
        return None
    texto = str(valor).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = texto.encode("ascii", "ignore").decode("ascii")
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto or None


def distancia_levenshtein(a, b):
    a = "" if a is None else str(a)
    b = "" if b is None else str(b)
    anterior = list(range(len(b) + 1))
    for i, caractere_a in enumerate(a, start=1):
        atual = [i]
        for j, caractere_b in enumerate(b, start=1):
            custo = 0 if caractere_a == caractere_b else 1
            atual.append(min(atual[-1] + 1, anterior[j] + 1, anterior[j - 1] + custo))
        anterior = atual
    return anterior[-1]


def similaridade_jaro_winkler(a, b):
    a = "" if a is None else str(a)
    b = "" if b is None else str(b)
    if a == b:
        return 1.0
    if not a or not b:
        return 0.0

    janela = max(max(len(a), len(b)) // 2 - 1, 0)
    pares_a = [False] * len(a)
    pares_b = [False] * len(b)
    correspondencias = 0

    for i, caractere in enumerate(a):
        inicio = max(0, i - janela)
        fim = min(i + janela + 1, len(b))
        for j in range(inicio, fim):
            if pares_b[j] or caractere != b[j]:
                continue
            pares_a[i] = True
            pares_b[j] = True
            correspondencias += 1
            break

    if correspondencias == 0:
        return 0.0

    caracteres_a = [c for i, c in enumerate(a) if pares_a[i]]
    caracteres_b = [c for i, c in enumerate(b) if pares_b[i]]
    transposicoes = sum(x != y for x, y in zip(caracteres_a, caracteres_b)) / 2
    jaro = (
        correspondencias / len(a)
        + correspondencias / len(b)
        + (correspondencias - transposicoes) / correspondencias
    ) / 3

    prefixo = 0
    for x, y in zip(a[:4], b[:4]):
        if x != y:
            break
        prefixo += 1
    return jaro + prefixo * 0.1 * (1 - jaro)


## Leitura e seleção das Mortes Violentas Intencionais


In [3]:
ro = pd.read_csv(ARQUIVO_ISP, low_memory=False)
ro["ano"] = pd.to_numeric(ro["ano"], errors="coerce").astype("Int64")
ro["titulo_do_norm"] = ro["titulo_do"].map(normalizar_texto)
ro["municipio_norm"] = ro["municipio_fato"].map(normalizar_texto)

categorias_mvi = {
    "homicidio doloso",
    "latrocinio roubo seguido de morte",
    "lesao corporal seguida de morte",
    "morte por intervencao de agente do estado",
}

mvi = ro.loc[
    ro["ano"].between(2019, 2024, inclusive="both")
    & ro["titulo_do_norm"].isin(categorias_mvi)
    & ro["municipio_norm"].fillna("").str.contains("rio de janeiro")
].copy()

print("Registros de MVI selecionados:", len(mvi))
print(mvi.groupby("ano").size())
assert len(mvi) == 8771




Registros de MVI selecionados: 8771
ano
2019    1913
2020    1420
2021    1306
2022    1319
2023    1438
2024    1375
dtype: int64


## Valores ausentes e grafias territoriais


In [4]:
valores_sem_informacao = {
    "",
    "na",
    "nan",
    "ni",
    "sem informacao",
    "nao informado",
    "ignorado",
    "nao se aplica",
    "bairro nao cadastrado",
    "ilha do governador",
    "indeterminado",
    "nao sabido",
    "nao identificado",
    "desconhecido",
}

mvi["bairro_fato_norm"] = mvi["bairro_fato"].map(normalizar_texto)
mvi["bairro_inicialmente_identificado"] = ~mvi["bairro_fato_norm"].isin(valores_sem_informacao)
mvi.loc[mvi["bairro_fato_norm"].isna(), "bairro_inicialmente_identificado"] = False

resumo_ausencia = pd.DataFrame(
    {
        "registros_mvi": [len(mvi)],
        "sem_bairro_identificado": [(~mvi["bairro_inicialmente_identificado"]).sum()],
        "com_bairro_identificado": [mvi["bairro_inicialmente_identificado"].sum()],
        "grafias_literais": [mvi["bairro_fato"].nunique(dropna=True)],
    }
)
resumo_ausencia


,registros_mvi,sem_bairro_identificado,com_bairro_identificado,grafias_literais
0,8771,1323,7448,392


## Pareamento dos nomes dos bairros


As decisões resultantes da revisão manual foram preservadas em `regras_correcao_bairros_isp.csv`. O arquivo torna a etapa auditável e evita que decisões territoriais permaneçam ocultas no código.

In [5]:
bairros = pd.read_csv(ARQUIVO_BAIRROS, encoding="utf-8-sig")
coluna_nome = "nome" if "nome" in bairros.columns else "bairro"
bairros = bairros[[coluna_nome]].rename(columns={coluna_nome: "bairro_oficial"})
bairros["chave"] = bairros["bairro_oficial"].map(normalizar_texto)
bairros = bairros.dropna().drop_duplicates("chave")
oficiais = bairros["chave"].tolist()
nome_oficial = dict(zip(bairros["chave"], bairros["bairro_oficial"]))

correcoes = pd.read_excel(ARQUIVO_CORRECOES)
correcoes["chave"] = correcoes["bairro_fato_original"].map(normalizar_texto)
correcoes["correcao_final"] = correcoes["Correção"].where(correcoes["Correção"].notna(), pd.NA)
correcoes["correcao_final"] = correcoes["correcao_final"].astype("string").str.strip()
mapa_manual = (
    correcoes.dropna(subset=["chave"])
    .drop_duplicates("chave", keep="last")
    .set_index("chave")["correcao_final"]
    .to_dict()
)

grafias = (
    mvi.loc[mvi["bairro_inicialmente_identificado"], ["bairro_fato_norm"]]
    .drop_duplicates()
    .rename(columns={"bairro_fato_norm": "chave"})
)

resultados = []
for chave in grafias["chave"]:
    distancias = {oficial: distancia_levenshtein(chave, oficial) for oficial in oficiais}
    similaridades = {oficial: similaridade_jaro_winkler(chave, oficial) for oficial in oficiais}
    melhor_lev = min(distancias, key=distancias.get)
    melhor_jw = max(similaridades, key=similaridades.get)
    resultados.append(
        {
            "chave": chave,
            "bairro_levenshtein": nome_oficial[melhor_lev],
            "distancia_levenshtein": distancias[melhor_lev],
            "bairro_jaro_winkler": nome_oficial[melhor_jw],
            "similaridade_jaro_winkler": similaridades[melhor_jw],
            "concordancia": melhor_lev == melhor_jw,
        }
    )

pareamento = pd.DataFrame(resultados)
regras_revisadas = pd.read_csv(ARQUIVO_REGRAS_REVISADAS)
mapa_revisado = dict(
    zip(regras_revisadas["chave"], regras_revisadas["bairro_correcao_final"])
)

pareamento["correcao_revisada"] = pareamento["chave"].map(mapa_revisado)
pareamento["correcao_manual"] = pareamento["chave"].map(mapa_manual)
pareamento["bairro_padronizado"] = pareamento["correcao_revisada"]
pareamento["bairro_padronizado"] = pareamento["bairro_padronizado"].fillna(
    pareamento["correcao_manual"]
)
sem_manual = pareamento["bairro_padronizado"].isna()
pareamento.loc[sem_manual & pareamento["concordancia"], "bairro_padronizado"] = (
    pareamento.loc[sem_manual & pareamento["concordancia"], "bairro_levenshtein"]
)

mvi = mvi.merge(pareamento, left_on="bairro_fato_norm", right_on="chave", how="left")
mvi.loc[~mvi["bairro_inicialmente_identificado"], "bairro_padronizado"] = pd.NA

print("Concordância entre algoritmos:")
print(pareamento["concordancia"].value_counts(dropna=False))
print("Registros com bairro padronizado:", mvi["bairro_padronizado"].notna().sum())


Concordância entre algoritmos:
concordancia
True     254
False     95
Name: count, dtype: int64
Registros com bairro padronizado: 7436


## Compatibilização territorial e agregação anual


Como algumas decisões foram tomadas caso a caso, a lista dos 7.430 registros mantidos depois da revisão também é preservada. Essa lista é usada na agregação final e evita transformar decisões manuais em regras automáticas não documentadas.

In [6]:
regras = pd.read_csv(ARQUIVO_REGRAS)
regras = regras[regras["fontes"].isin(["todas", "ISP"])].copy()
regras["origem_norm"] = regras["origem"].map(normalizar_texto)
regras["destino"] = regras["destino"].where(regras["destino"].notna(), pd.NA)

registros_revisados = pd.read_csv(ARQUIVO_REGISTROS_REVISADOS)
assert len(registros_revisados) == 7430

mapa_compatibilidade = dict(zip(regras["origem_norm"], regras["destino"]))
registros_revisados["bairro_padronizado_norm"] = (
    registros_revisados["bairro_padronizado"].map(normalizar_texto)
)
registros_revisados["bairro"] = registros_revisados["bairro_padronizado"]
registros_revisados["bairro"] = (
    registros_revisados["bairro_padronizado_norm"]
    .map(mapa_compatibilidade)
    .fillna(registros_revisados["bairro"])
)

excluir = set(regras.loc[regras["acao"] == "excluir", "origem_norm"])
registros_revisados.loc[
    registros_revisados["bairro_padronizado_norm"].isin(excluir), "bairro"
] = pd.NA

mvi_validos = registros_revisados[registros_revisados["bairro"].notna()].copy()
mvi_validos["ano"] = mvi_validos["ano"].astype(int)

isp_bairro_ano = (
    mvi_validos.groupby(["bairro", "ano"], as_index=False)
    .size()
    .rename(columns={"size": "n_mvi"})
    .sort_values(["bairro", "ano"])
)

registro_saida = mvi_validos[
    ["controle", "ano", "titulo", "titulo_do", "bairro_fato_original", "bairro"]
].copy()
registro_saida.to_csv(SAIDA_REGISTROS, index=False, encoding="utf-8-sig")
isp_bairro_ano.to_csv(SAIDA, index=False, encoding="utf-8-sig")

print("Registros válidos antes da compatibilização final:", len(registros_revisados))
print("Registros válidos após a compatibilização:", len(mvi_validos))
print("Total agregado:", int(isp_bairro_ano["n_mvi"].sum()))
print("Bairros com ocorrência:", isp_bairro_ano["bairro"].nunique())
assert int(isp_bairro_ano["n_mvi"].sum()) == 7419


Registros válidos antes da compatibilização final: 7430
Registros válidos após a compatibilização: 7419
Total agregado: 7419
Bairros com ocorrência: 154


## Auditoria


In [7]:
auditoria = {
    "registros_mvi": int(len(mvi)),
    "registros_sem_bairro_inicial": int((~mvi["bairro_inicialmente_identificado"]).sum()),
    "registros_com_bairro_inicial": int(mvi["bairro_inicialmente_identificado"].sum()),
    "registros_validos_antes_compatibilizacao_final": int(len(registros_revisados)),
    "registros_validos_base_final": int(len(mvi_validos)),
    "total_mvi_agregado": int(isp_bairro_ano["n_mvi"].sum()),
    "anos": sorted(isp_bairro_ano["ano"].unique().tolist()),
    "arquivo_saida": str(SAIDA.relative_to(PROJETO)),
}
SAIDA_AUDITORIA.write_text(
    json.dumps(auditoria, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
auditoria


{'registros_mvi': 8771,
 'registros_sem_bairro_inicial': 1323,
 'registros_com_bairro_inicial': 7448,
 'registros_validos_antes_compatibilizacao_final': 7430,
 'registros_validos_base_final': 7419,
 'total_mvi_agregado': 7419,
 'anos': [2019, 2020, 2021, 2022, 2023, 2024],
 'arquivo_saida': 'dados/intermediarios/isp_bairro_ano.csv'}